# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a demonstration for loading and exploring the FAIR^2 tabular dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json), allowing standard metadata and programmatic access.

In [ ]:
# Install mlcroissant if needed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant` and review the dataset's overview.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets in the dataset and their structure. 

For each record set, list the associated fields and columns along with their `@id`s.


In [ ]:
# List available record sets by their @id
record_sets = metadata.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")

# Optionally: inspect the first record set's fields/columns
if record_sets:
    rs = record_sets[0]
    print(f"\nFields in record set {rs.id}:")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"  - @id: {field.id}, name: {field.name if hasattr(field, 'name') else ''}")
    if hasattr(rs, 'columns'):
        print(f"\nColumns in record set {rs.id}:")
        for col in rs.columns:
            print(f"  - @id: {col.id}, name: {col.name if hasattr(col, 'name') else ''}")

## 3. Data Extraction
Extract data records from each record set and load them into pandas DataFrames, referencing by their `@id`.
Adjust the record set @ids as found in data overview step above.

In [ ]:
# Fill in the record set @ids from the previous cell
record_set_ids = [rs.id for rs in metadata.record_sets]
# If multiple record sets, you can optionally restrict to the relevant one for the main table
# For this dataset, assume main data is in the first record set

dataframes = {}

for record_set_id in record_set_ids:
    # Load records using mlcroissant (referenced by @id)
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

main_record_set_id = record_set_ids[0]
print(f"Loaded columns for record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

# Preview the top rows
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Perform data cleaning and basic statistics. 

Choose a relevant numeric field and a grouping field by their `@id`s based on above column listing. 
This typically includes normalization, filtering, and grouping.

In [ ]:
# Pick a numeric field (column @id) for analysis
print("Available columns:")
print(dataframes[main_record_set_id].columns.tolist())

# Example placeholders (Replace with actual @id from your dataset):
numeric_field_id = None
group_field_id = None

# Guess numeric field and group field by inspecting the columns
# You can set these manually, e.g.:
# numeric_field_id = '@id_of_numeric_field'  # e.g. 'Age', but use actual @id
# group_field_id = '@id_of_group_field'      # e.g. 'Sex', but use actual @id
for col in dataframes[main_record_set_id].columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if ('sex' in col.lower()) or ('gender' in col.lower()):
        group_field_id = col

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Remove NaN and filter for values above a threshold (e.g., age > 40 years)
threshold = 40

if numeric_field_id is not None:
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by e.g. sex/gender field (if present), show groupwise mean
    if group_field_id is not None and group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df)
else:
    print("No numeric field found for EDA. Please update numeric_field_id and group_field_id above.")

## 5. Visualization
Visualize the distribution of the selected numeric field and group comparisons (e.g., histogram of age by sex).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
if numeric_field_id is not None and numeric_field_id in dataframes[main_record_set_id]:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].astype(float), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot by group
if group_field_id is not None and numeric_field_id in dataframes[main_record_set_id]:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=dataframes[main_record_set_id][group_field_id], y=dataframes[main_record_set_id][numeric_field_id].astype(float))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated using `mlcroissant` to access, extract, and analyze the FAIR^2 colorectal cancer survivor dataset using Croissant metadata references.

We loaded the main record set, explored available fields (via their `@id`s), performed simple EDA (filtering and normalization), and visualized key attributes. For project-specific analysis, update the numeric and grouping field ids to those most relevant to your questions.

Refer to the [dataset's Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) for full documentation and reproducibility.